# Classifier head + fusion + LACR + loss
Contains: `ParametricHead` (X), the shared interaction function (`CrossInteractionLayer` + `SharedInteractionMLP` -> Y), `HybridFusion` (z), `LACRModule` (applied after the fused logits), the model container, and the losses (masked LDAM-ASL, bounded LDAM-ASL) with held-out masking.

**Input:** `encoder`, `anchor_bank`, `adjacency`, and `class_counts` [28] with the seen mask for the loss.

In [ ]:
# ===== IMPORTS =====
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
GOEMOTIONS_LABELS: List[str] = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral",
]
NUM_CLASSES: int = len(GOEMOTIONS_LABELS)  # 28


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def banner(title: str) -> None:
    print("\n" + "=" * 78 + f"\n{title}\n" + "=" * 78)

def resolve_held_out(names: Sequence[str]) -> List[int]:
    """Map held-out class names to column indices (validates names, rejects 'all classes')."""
    unknown = [n for n in names if n not in GOEMOTIONS_LABELS]
    assert not unknown, f"unknown held-out classes: {unknown}"
    idx = sorted({GOEMOTIONS_LABELS.index(n) for n in names})
    assert len(idx) < NUM_CLASSES, "at least one class must remain seen"
    return idx

def make_seen_mask(num_classes: int, held_out_classes: Optional[Sequence[int]] = None) -> torch.Tensor:
    """Float mask [C]: 1.0 for seen classes, 0.0 for held-out (zero-shot) classes."""
    mask = torch.ones(num_classes)
    if held_out_classes:
        mask[torch.as_tensor(list(held_out_classes), dtype=torch.long)] = 0.0
    return mask

def _in_notebook() -> bool:
    try:
        ip = get_ipython()  # type: ignore[name-defined]  # noqa: F821
    except NameError:
        return False
    return ip is not None and getattr(ip, "kernel", None) is not None

def _show_table(df: pd.DataFrame, title: Optional[str] = None, styler=None) -> None:
    """Rich HTML table inside a notebook, plain text elsewhere."""
    if title:
        print(title)
    if _in_notebook():
        from IPython.display import display
        display(styler if styler is not None else df)
    else:
        print(df.to_string(float_format=lambda v: f"{v:.4f}"))

In [ ]:
@dataclass
class Config:
    held_out_classes: List[str] = field(default_factory=list)  # names of unseen emotions, e.g. grief pride relief
    max_len: int = 64                        # max tokens per sentence (verification probe)
    seed: int = 42
    # ---- classifier head / fusion / LACR ----
    mlp_hidden: int = 512
    dropout: float = 0.3
    lacr_proj_mode: str = "dense"            # "dense" Linear(C, C) (masked for held-out) | "shared" scalar
    lacr_transpose_adj: bool = True          # Correlation = z @ A_corr^T
    lacr_ln_bias_init: float = -2.0          # initial LayerNorm bias of LACR (e.g. -2 for a sparse-label prior)
    # ---- loss ----
    loss_type: str = "ldam_asl"              # ldam_asl | ldam | asl | bce
    gamma_pos: float = 0.0
    gamma_neg: float = 2.0
    prob_margin: float = 0.05                # ASL probability shift m for negatives
    ldam_max_margin: float = 0.3             # largest LDAM margin (rarest seen class)
    loss_reduction: str = "batch_mean"       # batch_mean (sum over classes, mean over batch) | "mean"
    logit_clip: float = 8.0                  # soft bound c in c * tanh(z / c); 0 = off
    logit_penalty: float = 0.005             # L2 penalty on raw logits; 0 = off


cfg = Config()
set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class CrossInteractionLayer(nn.Module):
    """Vectorised u_i = [ h || A_i || (h * A_i) || Y_cos,i ]  ->  [B, C, 3D + 1]   (2305 for D = 768)."""

    def __init__(self, hidden_size: int = 768):
        super().__init__()
        self.hidden_size = hidden_size

    @property
    def out_dim(self) -> int:
        return 3 * self.hidden_size + 1

    def forward(self, h: torch.Tensor, A: torch.Tensor, y_cos: torch.Tensor) -> torch.Tensor:
        B, D = h.shape
        C = A.size(0)
        assert D == self.hidden_size and A.shape == (C, D), f"h {tuple(h.shape)} vs A {tuple(A.shape)}"
        assert y_cos.shape == (B, C), f"Y_cos must be [{B}, {C}], got {tuple(y_cos.shape)}"
        h_e = h.unsqueeze(1).expand(B, C, D)   # [B, C, D]
        A_e = A.unsqueeze(0).expand(B, C, D)   # [B, C, D]
        return torch.cat([h_e, A_e, h_e * A_e, y_cos.unsqueeze(-1)], dim=-1)  # [B, C, 3D+1]


class SharedInteractionMLP(nn.Module):
    """One MLP shared by all classes: u_i (3D+1) -> scalar logit.  [B, C, 3D+1] -> [B, C]."""

    def __init__(self, in_dim: int = 2305, hidden_dim: int = 512, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        return self.net(u).squeeze(-1)


class InteractionBranch(nn.Module):
    """Path 2: (h, A) -> (Y [B, C], Y_cos [B, C])."""

    def __init__(self, hidden_size: int = 768, mlp_hidden: int = 512, dropout: float = 0.1):
        super().__init__()
        self.cross = CrossInteractionLayer(hidden_size)
        self.mlp = SharedInteractionMLP(self.cross.out_dim, mlp_hidden, dropout)

    def forward(self, h: torch.Tensor, A: torch.Tensor, y_cos: Optional[torch.Tensor] = None):
        if y_cos is None:
            y_cos = F.normalize(h, p=2, dim=-1) @ F.normalize(A, p=2, dim=-1).t()
        return self.mlp(self.cross(h, A, y_cos)), y_cos


class ParametricHead(nn.Module):
    """Path 1: X = W_c h + b_c  ->  [B, C]."""

    def __init__(self, hidden_size: int = 768, num_classes: int = 28, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        return self.fc(self.dropout(h))


class HybridFusion(nn.Module):
    """z = X * mask + Y : seen classes use X + Y, held-out (zero-shot) classes use Y only."""

    def __init__(self, num_classes: int = 28, held_out_classes: Optional[Sequence[int]] = None):
        super().__init__()
        self.register_buffer("seen_mask", make_seen_mask(num_classes, held_out_classes))

    def forward(self, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        assert X.shape == Y.shape and X.size(1) == self.seen_mask.numel()
        return X * self.seen_mask + Y


class LACRModule(nn.Module):
    """
    Label-Aware Correlation Residual, applied strictly AFTER fusion, on logits z [B, C]:
        Correlation = z @ A_corr^T                (transpose_adj=True: each class takes a convex combination of its
                                                   neighbours' logits; rows of A_corr sum to 1)
        z_final     = LayerNorm(z + Residual),  Residual = Linear(Correlation)  (zero-initialised)
    proj_mode "dense": nn.Linear(C, C) whose rows/columns of held-out classes are masked (untrained weights can
                       never leak noise into zero-shot logits); "shared": one scalar weight/bias for all classes.
    The adjacency must already be zero on held-out rows/cols (checked at construction).
    """

    def __init__(self, adjacency: torch.Tensor, held_out_classes: Optional[Sequence[int]] = None,
                 proj_mode: str = "dense", transpose_adj: bool = True, zero_init: bool = True,
                 ln_bias_init: float = 0.0, eps: float = 1e-5):
        super().__init__()
        C = adjacency.size(0)
        assert adjacency.shape == (C, C) and proj_mode in ("dense", "shared")
        seen = make_seen_mask(C, held_out_classes)
        held = seen == 0
        A = adjacency.detach().float().clone()
        assert A[held].abs().sum() == 0 and A[:, held].abs().sum() == 0, "A_corr leaks held-out classes"
        assert torch.allclose(A[~held].sum(dim=1), torch.ones(int((~held).sum())), atol=1e-4), "seen rows must sum to 1"
        self.proj_mode, self.transpose_adj = proj_mode, transpose_adj
        self.register_buffer("adjacency", A)
        self.register_buffer("seen_mask", seen)
        self.proj = nn.Linear(C, C) if proj_mode == "dense" else nn.Linear(1, 1)
        if zero_init:
            nn.init.zeros_(self.proj.weight)
            nn.init.zeros_(self.proj.bias)
        self.norm = nn.LayerNorm(C, eps=eps)
        nn.init.constant_(self.norm.bias, ln_bias_init)

    def correlation(self, z: torch.Tensor) -> torch.Tensor:
        A = self.adjacency.t() if self.transpose_adj else self.adjacency
        return z @ A  # [B, C] x [C, C] -> [B, C]

    def residual(self, z: torch.Tensor) -> torch.Tensor:
        corr = self.correlation(z)
        if self.proj_mode == "dense":
            W = self.proj.weight * self.seen_mask.unsqueeze(1) * self.seen_mask.unsqueeze(0)
            res = F.linear(corr, W, self.proj.bias * self.seen_mask)
        else:
            res = self.proj(corr.unsqueeze(-1)).squeeze(-1)
        return res * self.seen_mask  # held-out classes receive exactly zero residual

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        assert z.dim() == 2 and z.size(1) == self.adjacency.size(0), f"LACR expects fused logits [B, C], got {tuple(z.shape)}"
        return self.norm(z + self.residual(z))


class SemanticAnchorEmotionModel(nn.Module):
    """input_ids/attention_mask -> h -> (Path 1: X) + (Path 2: Y_cos -> u -> Y) -> z -> LACR -> z_final [B, 28]."""

    def __init__(self, encoder: nn.Module, anchor_bank: nn.Module, adjacency: torch.Tensor,
                 num_classes: int = NUM_CLASSES, held_out_classes: Optional[Sequence[int]] = None,
                 mlp_hidden: int = 512, dropout: float = 0.1,
                 lacr_kwargs: Optional[dict] = None):
        super().__init__()
        hidden = encoder.hidden_size
        assert anchor_bank.A.shape == (num_classes, hidden), f"anchors {tuple(anchor_bank.A.shape)} vs D={hidden}"
        self.encoder = encoder
        self.anchor_bank = anchor_bank
        self.interaction = InteractionBranch(hidden, mlp_hidden, dropout)
        self.head = ParametricHead(hidden, num_classes, dropout)
        self.fusion = HybridFusion(num_classes, held_out_classes)
        self.lacr = LACRModule(adjacency, held_out_classes, **(lacr_kwargs or {}))

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> Dict[str, torch.Tensor]:
        h = self.encoder(input_ids, attention_mask)   # [B, D]
        A = self.anchor_bank.A                         # [C, D] fixed buffer, no gradient
        Y, Y_cos = self.interaction(h, A)              # [B, C], [B, C]
        X = self.head(h)                               # [B, C]
        z = self.fusion(X, Y)                          # [B, C]
        return {"h": h, "Y_cos": Y_cos, "X": X, "Y": Y, "z": z, "z_final": self.lacr(z)}

SemanticAnchorClassifier = SemanticAnchorEmotionModel   # the same class under the name used in the project brief

In [ ]:
## --- Masked LDAM + ASL loss ---
class MaskedLDAMASLLoss(nn.Module):
    """
    Unified multi-label loss on logits [B, C] with zero-shot masking.
 
      LDAM margins   : Delta_j = Cst / N_j^{1/4}, with Cst chosen so that max_j Delta_j = ldam_max_margin
                       (seen classes only). Subtracted from the logit of POSITIVE labels only:
                       z'_j = z_j - Delta_j * y_j      (harder to reach a given probability for rare classes)
      ASL            : p = sigmoid(z');  positives:  -(1-p)^{gamma+} * log(p)
                       negatives (probability shifting, hard thresholding):  p_m = max(p - m, 0),
                       loss = -p_m^{gamma-} * log(1 - p_m)   (very easy negatives with p < m give exactly zero loss)
      Masking        : columns of held-out classes are multiplied by 0 (no loss, no gradient).
 
    loss_type: "ldam_asl" (margins + asymmetric focusing), "ldam" (margins, plain BCE shape), "asl" (no margins),
               "bce" (plain masked BCE). reduction: "batch_mean" (sum over classes, mean over batch) | "mean".
    """
 
    def __init__(self, class_counts: Sequence[float], seen_mask: torch.Tensor, loss_type: str = "ldam_asl",
                 ldam_max_margin: float = 0.5, gamma_pos: float = 0.0, gamma_neg: float = 4.0,
                 prob_margin: float = 0.05, reduction: str = "batch_mean", eps: float = 1e-8):
        super().__init__()
        assert loss_type in ("ldam_asl", "ldam", "asl", "bce"), loss_type
        assert reduction in ("batch_mean", "mean"), reduction
        assert gamma_pos >= 0 and (gamma_neg == 0 or gamma_neg >= 1), "gamma_neg must be 0 or >= 1"
        assert 0.0 <= prob_margin < 1.0
        seen = seen_mask.float()
        counts = torch.as_tensor(np.asarray(class_counts), dtype=torch.float32).clamp(min=1.0)
        inv_quarter = counts.pow(-0.25)                                   # N_j^{-1/4}
        cst = ldam_max_margin / inv_quarter[seen.bool()].max()            # C in Delta_j = C / N_j^{1/4}
        margins = cst * inv_quarter * seen
        use_ldam = loss_type in ("ldam_asl", "ldam")
        use_asl = loss_type in ("ldam_asl", "asl")
        self.register_buffer("margins", margins if use_ldam else torch.zeros_like(margins))
        self.register_buffer("seen_mask", seen)
        self.gamma_pos = gamma_pos if use_asl else 0.0
        self.gamma_neg = gamma_neg if use_asl else 0.0
        self.prob_margin = prob_margin if use_asl else 0.0
        self.reduction, self.eps = reduction, eps
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        z = logits.float()
        y = targets.float()
        assert z.shape == y.shape and z.size(1) == self.seen_mask.numel(), (tuple(z.shape), tuple(y.shape))
        z_adj = z - self.margins.unsqueeze(0) * y                          # LDAM: margin on positives only
        p = torch.sigmoid(z_adj)
        loss_pos = y * F.logsigmoid(z_adj)
        if self.gamma_pos > 0:
            loss_pos = loss_pos * (1.0 - p).pow(self.gamma_pos)
        if self.prob_margin > 0:                                           # ASL probability shifting
            p_m = (p - self.prob_margin).clamp(min=0.0)
            log_neg = torch.log((1.0 - p_m).clamp(min=self.eps))
        else:
            p_m = p
            log_neg = F.logsigmoid(-z_adj)                                 # numerically stable log(1 - p)
        loss_neg = (1.0 - y) * log_neg
        if self.gamma_neg > 0:
            loss_neg = loss_neg * p_m.clamp(min=self.eps).pow(self.gamma_neg)
        loss = -(loss_pos + loss_neg) * self.seen_mask.unsqueeze(0)       # zero out held-out classes
        if self.reduction == "batch_mean":
            return loss.sum() / z.size(0)
        return loss.sum() / (z.size(0) * self.seen_mask.sum().clamp(min=1.0))


## --- Loss selector (reads cfg.loss_type) ---
def build_loss_fn(cfg: Config, class_counts: Sequence[float], seen_mask: torch.Tensor) -> nn.Module:
    """Selects the loss from cfg.loss_type: "ldam_asl" / "ldam" / "asl" / "bce" -> MaskedLDAMASLLoss.
    Receives the TRAIN class counts and the seen mask."""
    if cfg.loss_type in ("ldam_asl", "ldam", "asl", "bce"):
        return MaskedLDAMASLLoss(class_counts, seen_mask, cfg.loss_type, cfg.ldam_max_margin, cfg.gamma_pos,
                                 cfg.gamma_neg, cfg.prob_margin, cfg.loss_reduction)
    raise ValueError(f"unknown loss_type {cfg.loss_type!r}; expected ldam_asl | ldam | asl | bce")


def describe_loss(loss_fn: nn.Module, cfg: Config) -> str:
    """One-line summary printed at start-up (LDAM margins)."""
    seen = loss_fn.seen_mask > 0
    return (f"[LOSS] {cfg.loss_type} | LDAM margins min/max over seen classes: "
            f"{loss_fn.margins[seen].min():.3f}/{loss_fn.margins.max():.3f}")

In [ ]:
## --- Bounded LDAM-ASL (same maths as the masked LDAM-ASL + soft logit bound + logit penalty) ---
# (D) LDAM-ASL with bounded logits: same maths as MaskedLDAMASLLoss, plus
#     * soft bound   z_b = c * tanh(z / c)  -> the gradient of an over-confident logit decays smoothly,
#                    so the unbounded "-log p" pull on positives (gamma+ = 0) cannot inflate logits forever
#     * logit penalty lambda * mean_seen( z^2 )  -> explicit L2 on the raw logits (held-out columns excluded)
# ---------------------------------------------------------------------------------------------
class BoundedMaskedLDAMASLLoss(MaskedLDAMASLLoss):
    def __init__(self, *args, logit_clip: float = 8.0, logit_penalty: float = 0.005, **kwargs):
        super().__init__(*args, **kwargs)
        self.logit_clip, self.logit_penalty = float(logit_clip), float(logit_penalty)
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        z = logits.float()
        z_b = self.logit_clip * torch.tanh(z / self.logit_clip) if self.logit_clip > 0 else z
        loss = super().forward(z_b, targets)
        if self.logit_penalty > 0:
            pen = (z.pow(2) * self.seen_mask.unsqueeze(0)).sum() / (z.size(0) * self.seen_mask.sum().clamp(min=1.0))
            loss = loss + self.logit_penalty * pen
        return loss
 
 

In [ ]:
# ---------------- run: build the model and verify one forward pass ----------------
# needs: encoder, anchor_bank, adjacency (from the encoder / anchors file), tokenizer, and `train["text"]` for the probe
banner("MODEL")
held_out = resolve_held_out(cfg.held_out_classes)
model = SemanticAnchorClassifier(
    encoder, anchor_bank, adjacency, NUM_CLASSES, held_out, cfg.mlp_hidden, cfg.dropout,
    lacr_kwargs={"proj_mode": cfg.lacr_proj_mode, "transpose_adj": cfg.lacr_transpose_adj,
                 "ln_bias_init": cfg.lacr_ln_bias_init}).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[MODEL] trainable parameters: {n_params / 1e6:.1f}M | held-out classes: "
      f"{[GOEMOTIONS_LABELS[i] for i in held_out] or 'none (fully supervised)'}")

# Verification print: anchor dims and the shape of every output of one forward pass (4 train sentences)
probe = tokenizer(train["text"].head(4).tolist(), padding=True, truncation=True, max_length=cfg.max_len, return_tensors="pt")
model.eval()
with torch.no_grad():
    probe_out = model(probe["input_ids"].to(device), probe["attention_mask"].to(device))
model.train()
print(f"[VERIFY] anchors A {tuple(model.anchor_bank.A.shape)} | input_ids {tuple(probe['input_ids'].shape)}")
shape_table = pd.DataFrame({
    "output": list(probe_out.keys()),
    "shape": [str(tuple(v.shape)) for v in probe_out.values()],
    "meaning": ["pooled sentence embedding h", "cosine(h, A_i) per class", "parametric logits X", "interaction logits Y",
                "fused logits z = X*seen + Y", "after LACR (final logits)"],
})
_show_table(shape_table, "\nOutput shapes of one forward pass:")
if held_out:
    print(f"[VERIFY] held-out columns: X = {probe_out['X'][:, held_out].abs().max():.4f} (ignored by fusion) | "
          f"z uses Y only | LACR residual on held-out = {model.lacr.residual(probe_out['z'])[:, held_out].abs().max():.1f}")
assert probe_out["z_final"].shape == (probe["input_ids"].shape[0], NUM_CLASSES)
assert torch.isfinite(probe_out["z_final"]).all(), "non-finite logits"